<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A;
             padding-bottom:.15em; margin-bottom:.5em; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size:.62em; border-radius:6px; border:1px solid #E2E6EE; box-shadow:none; }
.reveal pre code { max-height:420px; }
.reveal ul li, .reveal ol li { margin-bottom:.3em; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
</style>

## Generación de Variables Aleatorias
### Modelado de Sistemas bajo Incertidumbre · Semana 2
---
Departamento de Ingeniería Industrial · Universidad de los Andes

## Agenda
1. El flujo de dos etapas: fuente → variada
2. Generador Congruencial Lineal — ejemplo numérico paso a paso
3. LCG: visualización y prueba de uniformidad
4. Teorema de la Transformada Inversa
5. Aplicación 1 — Tiempos de servicio: Exponencial (derivación + tabla numérica)
6. Aplicación 2 — Duración de actividades: Triangular
7. Caso: Auditoría del generador del Casino Andino

## El Flujo de Dos Etapas

Las computadoras son **deterministas** — dada la misma entrada, producen la misma salida. Sin embargo, la simulación requiere aleatoriedad. La solución son los **números pseudoaleatorios**.

<div class="defn">

**Etapa 1 — Fuente (materia prima):** generar $R_i \sim U(0,1)$ mediante un algoritmo aritmético.

**Etapa 2 — Variada (producto final):** transformar $R_i$ en $X_i \sim F$ mediante $X_i = F^{-1}(R_i)$.

</div>

$$U(0,1) \xrightarrow{\;\text{LCG}\;} R_1, R_2, R_3, \ldots \xrightarrow{\;F^{-1}(\cdot)\;} X_1, X_2, X_3, \ldots \sim F$$

**Propiedades deseables del generador de la Etapa 1:**
- Uniforme en $[0,1)$ — sin sesgos detectables
- **Período largo** — no repetirse pronto (Python usa Mersenne Twister: período $2^{19937}-1$)
- **Reproducible** — misma semilla → mismos números → experimentos verificables

## Generador Congruencial Lineal (GCL / LCG)

<div class="defn">

$$Z_i = (a \cdot Z_{i-1} + c) \;\bmod\; m, \qquad R_i = \frac{Z_i}{m}$$

| Parámetro | Rol |
|---|---|
| $Z_0$ | Semilla (valor inicial) |
| $a$ | Multiplicador |
| $c$ | Incremento |
| $m$ | Módulo — determina el **período máximo** |

</div>

**Ejemplo numérico de la lectura** — generador didáctico con $m=16,\; a=5,\; c=3,\; Z_0=7$:

$$Z_1 = (5 \times 7 + 3)\bmod 16 = 38\bmod 16 = 6 \quad\Rightarrow\quad R_1 = 6/16 = 0.3750$$
$$Z_2 = (5 \times 6 + 3)\bmod 16 = 33\bmod 16 = 1 \quad\Rightarrow\quad R_2 = 1/16 = 0.0625$$

El código siguiente genera y muestra el ciclo completo:

In [ ]:
# Ejemplo numérico exacto de la lectura: m=16, a=5, c=3, Z0=7
m, a, c, Z0 = 16, 5, 3, 7

print(f"LCG didáctico:  m={m}, a={a}, c={c}, Z₀={Z0}")
print("─" * 54)
print(f"{'i':>3}  {'Cálculo':>28}  {'Zᵢ':>4}  {'Rᵢ = Zᵢ/m':>11}")
print("─" * 54)

Z = Z0
for i in range(1, 20):
    Z_new = (a * Z + c) % m
    R     = Z_new / m
    calculo = f"({a}×{Z}+{c}) mod {m} = {a*Z+c} mod {m}"
    print(f"  {i:>2}  {calculo:>28}  {Z_new:>4}  {R:>11.4f}")
    Z = Z_new
    if Z_new == Z0:
        print(f"  → Ciclo completo en i={i}. Período = {i}.")
        break
print("─" * 54)
print(f"  Con m=2³² profesional, el período sería hasta {2**32:,} valores.")

In [ ]:
import matplotlib.pyplot as plt

# LCG con parámetros profesionales (Numerical Recipes)
def lcg(seed, a=1664525, c=1013904223, m=2**32, n=5000):
    X, out = seed, []
    for _ in range(n):
        X = (a * X + c) % m
        out.append(X / m)
    return out

R_lcg = lcg(seed=42)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(R_lcg, bins=40, color='#1A7A9A', edgecolor='white', alpha=0.85, density=True)
axes[0].axhline(1.0, color='#C8961E', lw=2.5, ls='--', label='PDF $U(0,1) = 1$')
axes[0].set_title('LCG profesional — distribución de $R_i$  ($n=5000$)')
axes[0].set_xlabel('$R_i$'); axes[0].set_yticks([])
axes[0].legend(fontsize=9)

axes[1].scatter(R_lcg[:-1], R_lcg[1:], s=0.8, c='#0D2240', alpha=0.3)
axes[1].set_title('Pares consecutivos $(R_i,\\ R_{i+1})$ — sin estructura')
axes[1].set_xlabel('$R_i$'); axes[1].set_ylabel('$R_{i+1}$')

plt.suptitle('Etapa 1 — Verificación visual del generador LCG', fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## Teorema de la Transformada Inversa

<div class="defn">

Si $X$ es una variable aleatoria continua con CDF $F(x)$, entonces $U = F(X) \sim U(0,1)$.

**Consecuencia (generación):** dado $R \sim U(0,1)$,

$$X = F^{-1}(R) \sim F$$

</div>

**Intuición geométrica:**

$$R \xrightarrow{\;\text{leer en eje } y\;} \text{CDF} \xrightarrow{\;F^{-1}\;} x \text{ en eje } x$$

Un valor $R$ pequeño (cercano a 0) mapea a la cola izquierda de $F$; un $R$ grande mapea a la cola derecha.

**Procedimiento para cualquier distribución continua:**
1. Obtener la CDF: $F(x) = \int_{-\infty}^x f(t)\,dt$
2. Igualar a $R$: $F(x) = R$
3. Despejar $x$: $x = F^{-1}(R)$

## Aplicación 1 — Tiempos de Servicio en un Cajero

**Contexto:** Un ingeniero simula un cajero automático. El tiempo de servicio sigue $\text{Exp}(\lambda=2)$ — media $= 0.5$ min/cliente.

**Derivación de $F^{-1}$:**
$$F(x) = 1 - e^{-\lambda x} = R \;\Longrightarrow\; X = -\frac{1}{\lambda}\ln(1-R) \approx -\frac{1}{\lambda}\ln R$$

**Aplicación directa con los $R_i$ del LCG didáctico** (de la lectura):

| Cliente | $R_i$ (del LCG) | Cálculo | $X_i$ (min) |
|---|---|---|---|
| 1 | $R_1 = 0.3750$ | $-0.5 \times \ln(0.3750) = -0.5\times(-0.9808)$ | **0.490 min** |
| 2 | $R_2 = 0.0625$ | $-0.5 \times \ln(0.0625) = -0.5\times(-2.7726)$ | **1.386 min** — cola larga |
| 3 | $R_3 = 0.5000$ | $-0.5 \times \ln(0.5000) = -0.5\times(-0.6931)$ | **0.347 min** |

<div class="nota">

$R_2 = 0.0625$ es muy pequeño → $X_2 = 1.39$ min, mucho mayor que la media. La transformada inversa captura automáticamente la **asimetría** de la exponencial.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import expon
np.random.seed(5)

lam, media = 2.0, 0.5   # λ=2, media=0.5 min

# --- Pipeline completo: LCG(toy) → R → X ---
m, a, c, Z = 16, 5, 3, 7
print("Pipeline LCG → Transformada Inversa → X ~ Exp(λ=2)")
print("─" * 58)
print(f"{'Cliente':>8}  {'Rᵢ (LCG)':>10}  {'Xᵢ = −0.5·ln(Rᵢ)':>20}  {'Tipo'}")
print("─" * 58)
for i in range(3):
    Z = (a * Z + c) % m
    R = Z / m
    X = -media * np.log(R)
    tipo = "servicio largo (cola)" if X > 2*media else ("servicio normal" if X > media else "servicio rápido")
    print(f"  {i+1:>6}  {R:>10.4f}  {X:>20.4f} min  ← {tipo}")
print("─" * 58)

# --- Verificación a gran escala: inversa manual vs numpy ---
U     = np.random.uniform(0, 1, 5_000)
X_inv = -media * np.log(U)                         # transformada inversa
X_np  = np.random.exponential(media, 5_000)         # NumPy interno

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(X_inv, bins=55, alpha=0.65, color='#1A7A9A',
        label='Transformada inversa  $X = -0.5\\ln R$', density=True)
ax.hist(X_np,  bins=55, alpha=0.50, color='#C8961E',
        label='np.random.exponential', density=True)
x_t = np.linspace(0, 4, 300)
ax.plot(x_t, expon.pdf(x_t, scale=media), 'k--', lw=2,
        label='PDF teórica  $f(x)=\\lambda e^{-\\lambda x}$')
ax.set_xlabel('Tiempo de servicio (min)'); ax.set_ylabel('Densidad')
ax.set_title(f'Etapa 2 — $\\mathrm{{Exp}}(\\lambda={lam})$ por transformada inversa  ($n=5000$)')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

## Aplicación 2 — Duración de Actividades: Triangular

**Contexto:** Al simular un proyecto de ingeniería, la duración de una actividad es incierta. Se conocen el mínimo ($a$), la moda ($m$) y el máximo ($b$): $T(a, m, b)$.

**Derivación de $F^{-1}$ (por tramos):**

$$F^{-1}(u) = \begin{cases} a + \sqrt{u\,(b-a)(m-a)} & 0 \le u \le F(m) \\[4pt] b - \sqrt{(1-u)(b-a)(b-m)} & F(m) < u \le 1 \end{cases}$$

donde $F(m) = \dfrac{m - a}{b - a}$ es la probabilidad acumulada en la moda.

<div class="nota">

La Triangular no tiene forma cerrada en **NumPy** para la transformada — se implementa manualmente. Es la distribución estándar para duraciones de actividades en simulación PERT.

</div>

**Ejemplo:** actividad cuya duración mínima es 2 días, moda 5 días, máximo 10 días → $T(2, 5, 10)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import triang
np.random.seed(7)

a, mode, b = 2, 5, 10   # mínimo, moda, máximo (días)
n = 8_000

# Transformada inversa para Triangular(a, mode, b)
def triangular_inv(u, a, mode, b):
    Fm = (mode - a) / (b - a)
    return np.where(
        u <= Fm,
        a + np.sqrt(u * (b - a) * (mode - a)),
        b - np.sqrt((1 - u) * (b - a) * (b - mode))
    )

U     = np.random.uniform(0, 1, n)
X_inv = triangular_inv(U, a, mode, b)
X_np  = np.random.triangular(a, mode, b, n)

# Verificar las primeras 5 conversiones
print(f"Distribución: T({a}, {mode}, {b}) — duración de actividad (días)")
print(f"{'u':>10}  {'F⁻¹(u) = Xᵢ':>14}")
print("─" * 28)
for u_val in [0.10, 0.30, (mode-a)/(b-a), 0.70, 0.95]:
    x_val = triangular_inv(np.array([u_val]), a, mode, b)[0]
    print(f"  u = {u_val:.2f}   →   {x_val:.3f} días")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(X_inv, bins=50, alpha=0.65, color='#1565C0',
        label='Transformada inversa', density=True)
ax.hist(X_np,  bins=50, alpha=0.50, color='#C8961E',
        label='np.random.triangular', density=True)
c_param = (mode - a) / (b - a)
x_t = np.linspace(a, b, 300)
ax.plot(x_t, triang.pdf(x_t, c=c_param, loc=a, scale=b-a), 'k--', lw=2, label='PDF teórica')
ax.axvline(mode, color='#6A1B9A', lw=2, ls=':', label=f'Moda = {mode} días')
ax.set_xlabel('Duración de la actividad (días)'); ax.set_ylabel('Densidad')
ax.set_title(f'$T({a},\\ {mode},\\ {b})$ — transformada inversa por tramos')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()
print(f"\n  E[X] ≈ {np.mean(X_inv):.2f} días   (teórico: {(a+mode+b)/3:.2f})   SD ≈ {np.std(X_inv):.2f} días")

## Caso: Auditoría del Generador — Casino Andino

**Situación:** El casino usa un generador electrónico de dados. Un auditor de ingeniería industrial sospecha que el dado está sesgado — la cara 6 aparece con más frecuencia de la esperada.

**Prueba $\chi^2$ de bondad de ajuste:**

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i} \;\sim\; \chi^2_{k-1} \quad \text{bajo } H_0$$

| | Descripción |
|---|---|
| $H_0$ | $p_i = 1/6$ para toda cara $i$ — generador uniforme |
| $H_1$ | Al menos una $p_i \neq 1/6$ — generador sesgado |
| Regla | Rechazar $H_0$ si $p\text{-valor} < \alpha = 0.05$ |

<div class="nota">

Esta prueba es la herramienta estándar para **auditar** si un generador de números aleatorios produce uniformidad. La misma lógica aplica a cualquier prueba de hipótesis sobre distribuciones discretas.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare
np.random.seed(99)

n_lanzamientos = 3_000
# El dado del casino: cara 6 tiene probabilidad doble
probs_casino = [1/7, 1/7, 1/7, 1/7, 1/7, 2/7]
dados = np.random.choice([1,2,3,4,5,6], size=n_lanzamientos, p=probs_casino)

observado = np.array([(dados == i).sum() for i in range(1, 7)])
esperado  = np.full(6, n_lanzamientos / 6)
stat, p_valor = chisquare(f_obs=observado, f_exp=esperado)

# --- Tabla de resultados ---
print(f"Auditoría: {n_lanzamientos:,} lanzamientos del generador del Casino Andino")
print("─" * 52)
print(f"{'Cara':>5}  {'Observado':>10}  {'Esperado':>10}  {'(O-E)²/E':>10}")
print("─" * 52)
for i, (o, e) in enumerate(zip(observado, esperado), 1):
    contrib = (o - e)**2 / e
    marca = " ← !" if i == 6 else ""
    print(f"    {i}  {o:>10}  {e:>10.0f}  {contrib:>10.2f}{marca}")
print("─" * 52)
print(f"  χ² = {stat:.2f},   p-valor = {p_valor:.6f}")
print()
if p_valor < 0.05:
    print(f"  → Se RECHAZA H₀ (p < 0.05). El dado NO es uniforme.")
    print(f"  → El auditor tiene evidencia estadística de fraude.")
else:
    print(f"  → No hay evidencia para rechazar H₀. El dado parece uniforme.")

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(7, 3))
caras = [f'Cara {i}' for i in range(1, 7)]
ax.bar(caras, observado, color=['#1A7A9A']*5 + ['#C62828'], edgecolor='white', label='Observado')
ax.axhline(esperado[0], color='#C8961E', lw=2, ls='--', label=f'Esperado = {esperado[0]:.0f}')
ax.set_ylabel('Frecuencia'); ax.set_title('Frecuencias observadas vs esperadas')
ax.legend(); plt.tight_layout(); plt.show()

## Conclusiones y Preguntas de Práctica

**Síntesis del flujo completo:**

$$\underbrace{Z_i = (aZ_{i-1}+c)\bmod m}_{\text{LCG (Etapa 1)}} \;\Rightarrow\; R_i = Z_i/m \;\Rightarrow\; \underbrace{X_i = F^{-1}(R_i)}_{\text{Transformada Inversa (Etapa 2)}}$$

- La **calidad** de la simulación depende del período del LCG y la exactitud de $F^{-1}$
- Cuando $F^{-1}$ no tiene forma cerrada (ej. Normal), se usan métodos alternativos (Box-Muller, Aceptación-Rechazo)
- La prueba $\chi^2$ **audita** que el generador produzca uniformidad

**Preguntas de práctica:**

1. Con el LCG de la lectura ($m=16, a=5, c=3, Z_0=7$), compute $R_4$ y $R_5$. Aplique la transformada inversa para obtener $X_4, X_5 \sim \text{Exp}(\lambda=2)$.

2. Derive $F^{-1}$ para la distribución $U(a,b)$ y verifique que produce `np.random.uniform(a,b)`.

3. Para $T(1, 4, 9)$: calcule $F(m)$ y aplique la transformada inversa a $u = 0.20$, $u = 0.50$, $u = 0.85$.

4. El Casino Andino repite su auditoría con solo $n=100$ lanzamientos. ¿Cambia la conclusión? ¿Por qué? (ejecute el código con `n_lanzamientos=100`).

5. ¿Cómo generaría $X \sim N(\mu, \sigma^2)$ por transformada inversa? ¿Cuál es la dificultad? (pista: `scipy.stats.norm.ppf`)